# Atlassian Jira MCP Client Demo

This notebook demonstrates how to load local `.env` credentials, instantiate the `AtlassianClient`, and call Jira endpoints directly.

In [ ]:
import sys
from pathlib import Path

# Append repository root to sys.path
sys.path.append("../..")

from mcp_servers.atlassian.app.config import ATLASSIAN_API_CONFIG
from mcp_servers.atlassian.app.security import create_atlassian_client
from mcp_servers.atlassian.app.schemas import (
    ListJiraProjectsRequest,
    SearchJiraIssuesRequest,
    ListJiraProjectCategoriesRequest
)

# Create client instance using Secret Manager/local dotenv configured credentials
client = create_atlassian_client()
print(f"Initialized client for instance: {client.instance_url}")

## 1. Discovering Available Projects

In [ ]:
req_projects = ListJiraProjectsRequest()
resp_projects = await client.list_projects(req_projects)

print(f"Execution status: {resp_projects.execution_status}")
print(f"Projects count: {len(resp_projects.projects)}")
print(resp_projects.projects)

## 2. Searching Jira Tickets using Bounded JQL

In [ ]:
# Using a bounded JQL search restriction
req_search = SearchJiraIssuesRequest(jql="updated >= -30d order by created DESC", max_results=5)
resp_search = await client.search_issues(req_search)

print(f"Execution status: {resp_search.execution_status}")
if resp_search.execution_status == "success":
    print(f"Found {len(resp_search.issues)} recent issues:")
    for issue in resp_search.issues:
        print(f"- [{issue.get('key')}] {issue.get('fields', {}).get('summary')}")
else:
    print(f"Error: {resp_search.execution_message}")

## 3. Retrieving Project Categories

In [ ]:
req_categories = ListJiraProjectCategoriesRequest()
resp_categories = await client.list_project_categories(req_categories)

print(f"Execution status: {resp_categories.execution_status}")
print(f"Categories found: {len(resp_categories.categories)}")
print(resp_categories.categories)